### Data Loading

In [20]:
%pip install -q torch_geometric

Note: you may need to restart the kernel to use updated packages.


In [21]:
import os
# Must be set before CUDA is initialised, for deterministic cuBLAS
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

import copy, json, math, random, sys, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
from torch_geometric.nn import GINEConv
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve
from tqdm.auto import tqdm

SEED = 42

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# warn_only: a non-deterministic op prints a warning instead of crashing
torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'python {sys.version.split()[0]} | torch {torch.__version__} | pyg {torch_geometric.__version__} '
      f'| numpy {np.__version__} | pandas {pd.__version__} | device {DEVICE}')

python 3.13.15 | torch 2.11.0+cu128 | pyg 2.8.1 | numpy 2.1.3 | pandas 2.3.3 | device cuda


In [22]:
CFG = dict(
    data_path='/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_Trans.csv',
    out_dir='/kaggle/working' if os.path.isdir('/kaggle/working') else '.',
    split=(0.6, 0.2, 0.2),          # train / val / test, by time
    hidden=64,
    num_layers=2,
    num_neighbors=[20, 10],         # len must equal num_layers
    dropout=0.1,
    batch_size=4096,                # seed transactions per training step
    eval_batch_size=8192,
    epochs=100,
    patience=3,                     # early stopping on val AP
    lr=5e-4,
    weight_decay=1e-5,
    pos_weight='sqrt',              # 'sqrt' -> sqrt(neg/pos), 'full' -> neg/pos, or a number
    seed=SEED,
)
assert len(CFG['num_neighbors']) == CFG['num_layers']
os.makedirs(CFG['out_dir'], exist_ok=True)
CFG

{'data_path': '/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_Trans.csv',
 'out_dir': '/kaggle/working',
 'split': (0.6, 0.2, 0.2),
 'hidden': 64,
 'num_layers': 2,
 'num_neighbors': [20, 10],
 'dropout': 0.1,
 'batch_size': 4096,
 'eval_batch_size': 8192,
 'epochs': 100,
 'patience': 3,
 'lr': 0.0005,
 'weight_decay': 1e-05,
 'pos_weight': 'sqrt',
 'seed': 42}

In [23]:
df = pd.read_csv(CFG['data_path'])
print(f'{len(df):,} transactions')
df.head()

5,078,345 transactions


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


In [24]:
acc_bank = pd.concat([
    df[['From Bank', 'Account']].set_axis(['bank', 'acc'], axis=1),
    df[['To Bank', 'Account.1']].set_axis(['bank', 'acc'], axis=1),
]).drop_duplicates()
n_acc = acc_bank['acc'].nunique()
n_multi = int((acc_bank.groupby('acc')['bank'].nunique() > 1).sum())
print(f'{n_multi:,} of {n_acc:,} account IDs ({n_multi / n_acc:.3%}) appear under more than one bank '
      f'and will be merged into a single node.')
del acc_bank

8 of 515,080 account IDs (0.002%) appear under more than one bank and will be merged into a single node.


In [25]:
df = df.drop(columns=['From Bank', 'To Bank', 'Payment Format'])
df = df.rename(columns={
    'Timestamp': 'ts', 'Account': 'src_acc', 'Account.1': 'dst_acc',
    'Amount Received': 'amt_recv', 'Receiving Currency': 'ccy_recv',
    'Amount Paid': 'amt_paid', 'Payment Currency': 'ccy_paid',
    'Is Laundering': 'y',
})
df['ts'] = pd.to_datetime(df['ts'], format='%Y/%m/%d %H:%M')
df['row_id'] = np.arange(len(df))
# Stable sort by time, tie-broken by original row order -> identical order on every run
df = df.sort_values(['ts', 'row_id'], kind='mergesort').reset_index(drop=True)
df['y'] = df['y'].astype(np.int8)
print(f"positives: {df['y'].sum():,} ({df['y'].mean():.4%})")
df.head()

positives: 5,177 (0.1019%)


,ts,src_acc,dst_acc,amt_recv,ccy_recv,amt_paid,ccy_paid,y,row_id
0,2022-09-01,8000F4670,8000F4670,14675.57,US Dollar,14675.57,US Dollar,0,2
1,2022-09-01,8005DFEB0,8005DFEB0,897.37,US Dollar,897.37,US Dollar,0,17
2,2022-09-01,8000F6850,8000F6850,99986.94,US Dollar,99986.94,US Dollar,0,158
3,2022-09-01,8001167D0,8001167D0,16.08,US Dollar,16.08,US Dollar,0,218
4,2022-09-01,8001364A0,8001364A0,10.30,US Dollar,10.30,US Dollar,0,281


In [26]:
daily = df.groupby(df['ts'].dt.date)['y'].agg(n='size', positives='sum', rate='mean')
daily['rate'] = daily['rate'].map('{:.4%}'.format)
daily

,n,positives,rate
ts,,,
2022-09-01,1114921,322,0.0289%
2022-09-02,754449,408,0.0541%
2022-09-03,207382,391,0.1885%
2022-09-04,207430,407,0.1962%
2022-09-05,482650,471,0.0976%
2022-09-06,482089,531,0.1101%
2022-09-07,482751,497,0.1030%
2022-09-08,482773,539,0.1116%
2022-09-09,654467,514,0.0785%


### Data Preprocessing

In [27]:
ts_ns = df['ts'].values.astype('int64')        # sorted ascending
n = len(df)
cut_idx = []
for q in np.cumsum(CFG['split'])[:-1]:
    t_cut = ts_ns[int(q * n)]
    cut_idx.append(int(np.searchsorted(ts_ns, t_cut, side='left')))

split = np.zeros(n, dtype=np.int8)
split[cut_idx[0]:cut_idx[1]] = 1
split[cut_idx[1]:] = 2
df['split'] = split

summary = df.groupby('split').agg(n=('y', 'size'), positives=('y', 'sum'), rate=('y', 'mean'),
                                  start=('ts', 'min'), end=('ts', 'max'))
summary.index = ['train', 'val', 'test']
summary['share'] = summary['n'] / n
print(summary.to_string(formatters={'rate': '{:.4%}'.format, 'share': '{:.1%}'.format}))

# Hard guarantee: no time overlap between splits
assert summary['end'].iloc[0] < summary['start'].iloc[1]
assert summary['end'].iloc[1] < summary['start'].iloc[2]

             n  positives    rate               start                 end share
train  3046861       2297 0.0754% 2022-09-01 00:00:00 2022-09-06 13:35:00 60.0%
val    1015602       1082 0.1065% 2022-09-06 13:36:00 2022-09-08 16:11:00 20.0%
test   1015882       1798 0.1770% 2022-09-08 16:12:00 2022-09-18 16:18:00 20.0%


### 5. Node IDs
Accounts are mapped to integer node IDs. This uses only account strings (no labels or amounts), so mapping over all splits leaks nothing. Nodes have no input features: every node starts from the same learned vector, and the GNN builds its representation from the transactions around it.

In [28]:
accounts = pd.unique(np.concatenate([df['src_acc'].values, df['dst_acc'].values]))
acc_index = pd.Index(accounts)
df['src'] = acc_index.get_indexer(df['src_acc']).astype(np.int64)
df['dst'] = acc_index.get_indexer(df['dst_acc']).astype(np.int64)
NUM_NODES = len(accounts)
print(f'{NUM_NODES:,} accounts (nodes)')

515,080 accounts (nodes)


### 6. Edge features (fitted on train rows only)

| Feature | Why |
|---|---|
| `log_paid`, `log_recv` (standardised) | Amounts are heavy-tailed |
| `z_paid`, `z_recv` | Amount relative to its own currency (1 BTC ≠ 1 Yen) |
| `fx_log_ratio` | Gap between paid and received: conversion or fee |
| `cross_ccy`, `self_loop` | Currency hop; account paying itself |
| `hour_sin`, `hour_cos` | Time of day, cyclic |
| one-hot `ccy_paid`, `ccy_recv` | Which assets are involved |

Absolute timestamp is deliberately **not** a feature: test times lie outside the training range.

In [29]:
is_train = df['split'].values == 0
feat = pd.DataFrame(index=df.index)

for side in ['paid', 'recv']:
    logv = np.log1p(df[f'amt_{side}'].astype(np.float64))
    ccy = df[f'ccy_{side}']
    g_mu, g_sd = logv[is_train].mean(), logv[is_train].std()
    stats = logv[is_train].groupby(ccy[is_train]).agg(['mean', 'std'])
    mu = ccy.map(stats['mean']).fillna(g_mu)
    sd = ccy.map(stats['std']).fillna(g_sd).replace(0, g_sd)   # unseen / single-row currencies -> global stats
    feat[f'log_{side}'] = (logv - g_mu) / g_sd
    feat[f'z_{side}'] = (logv - mu) / sd

ratio = np.log1p(df['amt_recv']) - np.log1p(df['amt_paid'])
feat['fx_log_ratio'] = (ratio - ratio[is_train].mean()) / (ratio[is_train].std() + 1e-9)
feat['cross_ccy'] = (df['ccy_paid'] != df['ccy_recv']).astype(np.float32)
feat['self_loop'] = (df['src'] == df['dst']).astype(np.float32)
hour = df['ts'].dt.hour + df['ts'].dt.minute / 60.0
feat['hour_sin'] = np.sin(2 * np.pi * hour / 24)
feat['hour_cos'] = np.cos(2 * np.pi * hour / 24)

currencies = sorted(set(df['ccy_paid']) | set(df['ccy_recv']))   # names only, no labels
for side in ['paid', 'recv']:
    oh = pd.get_dummies(pd.Categorical(df[f'ccy_{side}'], categories=currencies), prefix=f'ccy_{side}', dtype=np.float32)
    oh.index = df.index
    feat = pd.concat([feat, oh], axis=1)

FEATURES = list(feat.columns)
X = feat.to_numpy(dtype=np.float32)
assert np.isfinite(X).all(), 'non-finite edge features'
print(f'{X.shape[1]} edge features:', FEATURES)

39 edge features: ['log_paid', 'z_paid', 'log_recv', 'z_recv', 'fx_log_ratio', 'cross_ccy', 'self_loop', 'hour_sin', 'hour_cos', 'ccy_paid_Australian Dollar', 'ccy_paid_Bitcoin', 'ccy_paid_Brazil Real', 'ccy_paid_Canadian Dollar', 'ccy_paid_Euro', 'ccy_paid_Mexican Peso', 'ccy_paid_Ruble', 'ccy_paid_Rupee', 'ccy_paid_Saudi Riyal', 'ccy_paid_Shekel', 'ccy_paid_Swiss Franc', 'ccy_paid_UK Pound', 'ccy_paid_US Dollar', 'ccy_paid_Yen', 'ccy_paid_Yuan', 'ccy_recv_Australian Dollar', 'ccy_recv_Bitcoin', 'ccy_recv_Brazil Real', 'ccy_recv_Canadian Dollar', 'ccy_recv_Euro', 'ccy_recv_Mexican Peso', 'ccy_recv_Ruble', 'ccy_recv_Rupee', 'ccy_recv_Saudi Riyal', 'ccy_recv_Shekel', 'ccy_recv_Swiss Franc', 'ccy_recv_UK Pound', 'ccy_recv_US Dollar', 'ccy_recv_Yen', 'ccy_recv_Yuan']


In [30]:
# Global tensors on the device. Row i = transaction i (time-sorted).
SRC = torch.from_numpy(df['src'].to_numpy(copy=True)).to(DEVICE)
DST = torch.from_numpy(df['dst'].to_numpy(copy=True)).to(DEVICE)
EDGE_X = torch.from_numpy(X).to(DEVICE)
Y = torch.from_numpy(df['y'].values.astype(np.float32)).to(DEVICE)
SPLIT = torch.from_numpy(df['split'].to_numpy(copy=True)).to(DEVICE)

IDX = {name: torch.nonzero(SPLIT == k).squeeze(1) for k, name in enumerate(['train', 'val', 'test'])}
{k: len(v) for k, v in IDX.items()}

{'train': 3046861, 'val': 1015602, 'test': 1015882}

### 7. Graph + seeded neighbour sampler
For each split, the visible graph holds both directions of every transaction: the real direction (flag 0) and a reversed copy (flag 1). Sampling incoming edges in this graph therefore picks up both a node's senders and its receivers, which fan-in and fan-out patterns need.

Sampling per batch:
1. Start from the sender and receiver of each seed transaction.
2. Per hop, take all incoming edges of nodes with at most `k` of them; for busier nodes draw `k` at random.
3. Expand only newly reached nodes at the next hop.

All randomness comes from a `torch.Generator` seeded per epoch, so the same seed gives the same subgraphs.

In [31]:
class BiGraphSampler:
    def __init__(self, edge_ids: torch.Tensor, num_nodes: int, device: torch.device):
        s, d, x = SRC[edge_ids], DST[edge_ids], EDGE_X[edge_ids]
        src = torch.cat([s, d])
        dst = torch.cat([d, s])
        flag = torch.cat([torch.zeros(len(s), 1, device=device), torch.ones(len(s), 1, device=device)])
        ex = torch.cat([torch.cat([x, x]), flag], dim=1)

        order = torch.argsort(dst, stable=True)          # CSR keyed by target node
        self.src, self.dst, self.ex = src[order], dst[order], ex[order]
        deg = torch.bincount(dst, minlength=num_nodes)
        self.rowptr = torch.zeros(num_nodes + 1, dtype=torch.long, device=device)
        self.rowptr[1:] = torch.cumsum(deg, 0)
        self.device = device
        self._map = torch.full((num_nodes,), -1, dtype=torch.long, device=device)
        self.num_edges = len(src)

    def _in_edges(self, nodes, k, gen):
        start = self.rowptr[nodes]
        deg = self.rowptr[nodes + 1] - start
        small = deg <= k
        parts = []
        s_start, s_deg = start[small], deg[small]
        total = int(s_deg.sum())
        if total > 0:   # take every edge of low-degree nodes
            offs = torch.arange(total, device=self.device) - torch.repeat_interleave(torch.cumsum(s_deg, 0) - s_deg, s_deg)
            parts.append(torch.repeat_interleave(s_start, s_deg) + offs)
        l_start, l_deg = start[~small], deg[~small]
        if len(l_start) > 0:   # k uniform draws for busy nodes, duplicates removed by unique()
            r = torch.rand(len(l_start) * k, generator=gen, device=self.device)
            l_deg_k = l_deg.repeat_interleave(k)
            pick = torch.minimum((r * l_deg_k).long(), l_deg_k - 1)
            parts.append(l_start.repeat_interleave(k) + pick)
        if not parts:
            return torch.empty(0, dtype=torch.long, device=self.device)
        return torch.unique(torch.cat(parts))

    @torch.no_grad()
    def sample(self, seed_src, seed_dst, num_neighbors, gen):
        frontier = torch.unique(torch.cat([seed_src, seed_dst]))
        visited = frontier
        hops = []
        for k in num_neighbors:
            if frontier.numel() == 0:
                break
            e = self._in_edges(frontier, k, gen)
            hops.append(e)
            nbrs = torch.unique(self.src[e])
            frontier = nbrs[~torch.isin(nbrs, visited)]
            visited = torch.cat([visited, frontier])
        e = torch.cat(hops) if hops else torch.empty(0, dtype=torch.long, device=self.device)

        self._map[visited] = torch.arange(len(visited), device=self.device)
        edge_index = torch.stack([self._map[self.src[e]], self._map[self.dst[e]]])
        s_loc, d_loc = self._map[seed_src], self._map[seed_dst]
        self._map[visited] = -1
        return edge_index, self.ex[e], len(visited), s_loc, d_loc

### 8. Model: GINE with reverse message passing
- Every node starts from one learned vector, so differences between nodes come only from their transactions.
- Each layer runs two `GINEConv`s with separate weights: one over real-direction edges (what an account *received*) and one over reversed edges (what it *sent*). Results are summed, then LayerNorm, ReLU, dropout and a residual connection are applied.
- The transaction score uses `[h_sender, h_receiver, encoded transaction features]` → MLP → logit.

In [32]:
def mlp(i, h, o):
    return nn.Sequential(nn.Linear(i, h), nn.ReLU(), nn.Linear(h, o))

class EdgeGNN(nn.Module):
    def __init__(self, edge_dim, hidden, num_layers, dropout):
        super().__init__()
        self.node_init = nn.Parameter(torch.randn(1, hidden) * 0.1)
        self.edge_enc = mlp(edge_dim + 1, hidden, hidden)          # +1 = direction flag
        self.conv_in = nn.ModuleList([GINEConv(mlp(hidden, hidden, hidden), train_eps=True) for _ in range(num_layers)])
        self.conv_out = nn.ModuleList([GINEConv(mlp(hidden, hidden, hidden), train_eps=True) for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(num_layers)])
        self.head = nn.Sequential(nn.Linear(3 * hidden, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
        self.dropout = dropout

    def forward(self, num_nodes, edge_index, edge_x, s_loc, d_loc, seed_x):
        h = self.node_init.expand(num_nodes, -1)
        e = self.edge_enc(edge_x)
        fwd = edge_x[:, -1] == 0
        ei_f, e_f, ei_r, e_r = edge_index[:, fwd], e[fwd], edge_index[:, ~fwd], e[~fwd]
        for conv_i, conv_o, norm in zip(self.conv_in, self.conv_out, self.norms):
            m = conv_i(h, ei_f, e_f) + conv_o(h, ei_r, e_r)
            h = h + F.dropout(F.relu(norm(m)), p=self.dropout, training=self.training)
        seed_e = self.edge_enc(torch.cat([seed_x, torch.zeros(len(seed_x), 1, device=seed_x.device)], dim=1))
        return self.head(torch.cat([h[s_loc], h[d_loc], seed_e], dim=1)).squeeze(-1)

### 9. Training utilities
- **Class imbalance:** `pos_weight` up-weights positive transactions in the loss. The default `sqrt(neg/pos)` is gentler than the full ratio (~1000× here), which tends to make training unstable. Because AP depends only on ranking, the weight matters less than it would for accuracy.
- **Metrics:** AP is the headline metric (as on the leaderboard). The F1 threshold is chosen on **validation** and then applied unchanged to test.

In [33]:
def make_gen(seed):
    g = torch.Generator(device=DEVICE)
    g.manual_seed(seed)
    return g

def run_batches(model, sampler, idx, batch_size, gen, optimizer=None, criterion=None, shuffle=False, desc=None):
    training = optimizer is not None
    model.train(training)
    order = idx[torch.randperm(len(idx), generator=gen, device=DEVICE)] if shuffle else idx
    probs, total_loss, seen = [], 0.0, 0
    bar = tqdm(range(0, len(order), batch_size), desc=desc, unit='batch', leave=False, mininterval=1.0)
    with torch.set_grad_enabled(training):
        for b in bar:
            seeds = order[b:b + batch_size]
            ei, ex, n_nodes, s_loc, d_loc = sampler.sample(SRC[seeds], DST[seeds], CFG['num_neighbors'], gen)
            logit = model(n_nodes, ei, ex, s_loc, d_loc, EDGE_X[seeds])
            if training:
                loss = criterion(logit, Y[seeds])
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * len(seeds)
                seen += len(seeds)
                bar.set_postfix(loss=f'{total_loss / seen:.4f}', refresh=False)
            else:
                probs.append(torch.sigmoid(logit).float().cpu())
    bar.close()
    if training:
        return total_loss / len(order)
    return torch.cat(probs).numpy()

def metrics(y, p, thr=None, top_frac=0.01):
    out = {'AP': average_precision_score(y, p), 'ROC_AUC': roc_auc_score(y, p)}
    prec, rec, th = precision_recall_curve(y, p)
    if thr is None:   # pick the F1-optimal threshold (used on validation only)
        f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
        thr = float(th[np.argmax(f1)])
    pred = p >= thr
    tp = int((pred & (y == 1)).sum())
    out['threshold'] = thr
    out['precision@thr'] = tp / max(int(pred.sum()), 1)
    out['recall@thr'] = tp / max(int(y.sum()), 1)
    out['F1@thr'] = 2 * out['precision@thr'] * out['recall@thr'] / max(out['precision@thr'] + out['recall@thr'], 1e-12)
    k = max(1, int(round(top_frac * len(p))))
    top = np.argsort(-p, kind='stable')[:k]
    out[f'precision@top{top_frac:.0%}'] = float(y[top].mean())
    out[f'recall@top{top_frac:.0%}'] = float(y[top].sum() / max(y.sum(), 1))
    return out

def fmt(m):
    return ' | '.join(f'{k} {v:.4f}' for k, v in m.items())

### 10. Train (graph = train period only), validate (graph = train + val period)

In [34]:
set_seed(CFG['seed'])

train_sampler = BiGraphSampler(IDX['train'], NUM_NODES, DEVICE)
val_sampler = BiGraphSampler(torch.cat([IDX['train'], IDX['val']]), NUM_NODES, DEVICE)
print(f"train graph: {train_sampler.num_edges:,} directed edges | val graph: {val_sampler.num_edges:,}")

y_tr = Y[IDX['train']]
n_pos, n_neg = int(y_tr.sum()), int(len(y_tr) - y_tr.sum())
ratio = n_neg / max(n_pos, 1)
pw = {'sqrt': math.sqrt(ratio), 'full': ratio}.get(CFG['pos_weight'], CFG['pos_weight'])
print(f'train positives {n_pos:,} / negatives {n_neg:,} -> pos_weight {float(pw):.1f}')

model = EdgeGNN(len(FEATURES), CFG['hidden'], CFG['num_layers'], CFG['dropout']).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(pw), device=DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
print(f'{sum(p.numel() for p in model.parameters()):,} parameters')

y_val = df.loc[df['split'] == 1, 'y'].to_numpy()
best_ap, best_state, best_epoch, bad, history = -1.0, None, -1, 0, []

for epoch in range(1, CFG['epochs'] + 1):
    t0 = time.time()
    loss = run_batches(model, train_sampler, IDX['train'], CFG['batch_size'], make_gen(CFG['seed'] + epoch),
                       optimizer=optimizer, criterion=criterion, shuffle=True,
                       desc=f'epoch {epoch:02d}/{CFG["epochs"]} train')
    p_val = run_batches(model, val_sampler, IDX['val'], CFG['eval_batch_size'], make_gen(CFG['seed'] + 10_000),
                        desc=f'epoch {epoch:02d}/{CFG["epochs"]} val')
    val_ap = average_precision_score(y_val, p_val)
    history.append({'epoch': epoch, 'loss': loss, 'val_AP': val_ap})
    improved = val_ap > best_ap
    if improved:
        best_ap, best_epoch, bad = val_ap, epoch, 0
        best_state = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
    else:
        bad += 1
    print(f'epoch {epoch:02d} | loss {loss:.4f} | val AP {val_ap:.4f} {"*" if improved else ""} | {time.time() - t0:.0f}s')
    if bad >= CFG['patience']:
        print(f'early stop: no val AP gain for {CFG["patience"]} epochs')
        break

torch.save(best_state, os.path.join(CFG['out_dir'], 'best_model.pt'))
print(f'best val AP {best_ap:.4f} at epoch {best_epoch}')

train graph: 6,093,722 directed edges | val graph: 8,124,926
train positives 2,297 / negatives 3,044,564 -> pos_weight 36.4
52,805 parameters


epoch 01/100 train:   0%|          | 0/744 [00:00<?, ?batch/s]

epoch 01/100 val:   0%|          | 0/124 [00:00<?, ?batch/s]

epoch 01 | loss 0.1000 | val AP 0.1059 * | 45s


epoch 02/100 train:   0%|          | 0/744 [00:00<?, ?batch/s]

...

epoch 07/100 val:   0%|          | 0/124 [00:00<?, ?batch/s]

epoch 07 | loss 0.0530 | val AP 0.3744  | 44s


epoch 08/100 train:   0%|          | 0/744 [00:00<?, ?batch/s]

epoch 08/100 val:   0%|          | 0/124 [00:00<?, ?batch/s]

epoch 08 | loss 0.0525 | val AP 0.3655  | 44s
early stop: no val AP gain for 3 epochs
best val AP 0.3784 at epoch 5


### 11. Final evaluation: threshold from validation, applied to test (graph = all periods)

In [35]:
model.load_state_dict(best_state)
p_val = run_batches(model, val_sampler, IDX['val'], CFG['eval_batch_size'], make_gen(CFG['seed'] + 10_000),
                    desc='val (best model)')
val_m = metrics(y_val, p_val)

del train_sampler, val_sampler
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()

test_sampler = BiGraphSampler(torch.arange(len(df), device=DEVICE), NUM_NODES, DEVICE)
y_test = df.loc[df['split'] == 2, 'y'].to_numpy()
p_test = run_batches(model, test_sampler, IDX['test'], CFG['eval_batch_size'], make_gen(CFG['seed'] + 20_000),
                     desc='test')
test_m = metrics(y_test, p_test, thr=val_m['threshold'])

print('VAL :', fmt(val_m))
print('TEST:', fmt(test_m))
print(f"test positive rate {y_test.mean():.4%} -> AP of a random ranking ≈ {y_test.mean():.4f}")

val (best model):   0%|          | 0/124 [00:00<?, ?batch/s]

test:   0%|          | 0/125 [00:00<?, ?batch/s]

VAL : AP 0.3784 | ROC_AUC 0.9701 | threshold 0.8887 | precision@thr 0.6282 | recall@thr 0.3170 | F1@thr 0.4214 | precision@top1% 0.0741 | recall@top1% 0.6959
TEST: AP 0.4171 | ROC_AUC 0.9653 | threshold 0.8887 | precision@thr 0.6287 | recall@thr 0.3409 | F1@thr 0.4421 | precision@top1% 0.1236 | recall@top1% 0.6986
test positive rate 0.1770% -> AP of a random ranking ≈ 0.0018


### 12. Save run record
Seeds, config, library versions and results in one JSON file, which the hackathon's reproducibility rule asks for.

In [36]:
record = {
    'config': CFG, 'features': FEATURES, 'num_nodes': NUM_NODES,
    'split_sizes': {k: len(v) for k, v in IDX.items()},
    'best_epoch': best_epoch, 'history': history,
    'val': val_m, 'test': test_m,
    'versions': {'python': sys.version.split()[0], 'torch': torch.__version__,
                 'torch_geometric': torch_geometric.__version__, 'numpy': np.__version__,
                 'pandas': pd.__version__, 'device': str(DEVICE)},
}
path = os.path.join(CFG['out_dir'], 'run_record.json')
with open(path, 'w') as f:
    json.dump(record, f, indent=2, default=str)
print('saved', path)
pd.DataFrame(history)

saved /kaggle/working/run_record.json


,epoch,loss,val_AP
0,1,0.099978,0.105877
1,2,0.069347,0.277282
2,3,0.062959,0.245919
3,4,0.058537,0.344266
4,5,0.056477,0.378423
5,6,0.054160,0.362351
6,7,0.053044,0.374400
7,8,0.052476,0.365479
